# OCR a ticker's filings on a Kaggle T4 — the control notebook

**This notebook runs HERE, on your machine.** It builds the job, ships the filings, starts the
Kaggle kernel, waits for it and pulls the run folder back into `reports/pdf_ocr/`. The notebook
that runs *on Kaggle* is `src/web_scraper/RUN__pdf_ocr.ipynb` and you never open it — `kgpu`
patches its parameter cell and uploads it.

Edit **cell 1** and run the notebook top to bottom. Nothing else needs editing, and nothing is
written to `kaggle_config.json`.

⚠️ **IT WRITES NO STATEMENT CSV.** The output is a run folder that *scores itself* against the
CSVs already on disk. Merging a recovered quarter back into `raw_data/` stays a deliberate
Dagster act with a pre-run backup — CLAUDE.md §6-2-sexvicies has why, and it is four measured
silent downgrades.

The full guide, including what to do with the result, is `kgpu/PDF_OCR.md`.

In [ ]:
# ── PARAMETERS — the only cell you edit ───────────────────────────────────────
SYMBOL   = "VIC"          # ticker, as CafeF files it
EXCHANGE = "HOSE"         # HOSE | HNX | UPCOM

# WHICH FILINGS. Both filters are optional and they INTERSECT.
#   QUARTERS = ["2014-Q3"]              -> one quarter
#   QUARTERS = ["2013-Q4", "2014-Q1"]   -> a batch, in any order
#   QUARTERS = []  or  None            -> EVERY quarter this ticker files  (⚠️ 70+ documents)
#   PERIODS  = ["Q3-2014"]             -> the repo-native form, for `periods` only
# ⚠️ QUARTERS is YYYY-QQ because that form SORTS; the repo-native "Q3-2014" is REFUSED
QUARTERS = ["2014-Q3"]
PERIODS  = None

# ⚠️ TEMPLATE: None RESOLVES it (templates.csv, then CafeF's own fingerprint) and the run
# records WHICH route answered. State it ("bank" | "corp" | "securities" | "insurance") only
# to force both machines down one path when you are comparing them.
TEMPLATE = "corp"

ALLOW_PARENT = False      # fall back to the STANDALONE filing where no consolidated one exists
COMPARE      = True       # score every parsed cell against the statement CSV on disk
LAYERS       = None       # None = the full 47-layer cascade
NOTES        = ""         # free text into the run folder; blank writes a sensible default

# What to do. REHEARSE costs no Kaggle quota and catches a broken payload in ~60 s.
REHEARSE = True
RUN      = True           # False = build and inspect only, nothing is pushed

# UPSERT THE RESULT INTO raw_data/.../statements/*.csv AFTER THE PULL — on by default.
#    The merge runs HERE, not on the worker: a Kaggle kernel has no path to this disk.
#    It backs the three CSVs up first, prints every changed cell, and REFUSES three things
#    it cannot judge — a cumulative income statement, a statement whose `sane` band was
#    empty, and a figure that DIFFERS from a good `pdf` row already on disk.
# ⚠️ A statement a WORKER accepts is not always one a full run would: `sane`'s band is
#    rebuilt from disk here and accumulated in-run there, so the two gates can disagree
#    (measured on VIC Q3-2014). Read a RECOVERED figure against the filing before quoting
#    it. `kgpu merge <job> --dry-run` shows the decisions without writing.
MERGE_INTO_CSV = True

In [ ]:
# ── SETUP — imports and the working directory ─────────────────────────────────
# ⚠️ `kgpu` resolves the repo root from its own file, but the Kaggle client and the payload
# staging both use the CWD, so the notebook anchors itself the same way a shell would.
import os
import sys
from pathlib import Path

HERE = Path.cwd()
if HERE.name != "kaggle_gpu":
    candidate = next((p for p in [HERE / "src" / "kaggle_gpu", HERE / "kaggle_gpu"]
                      if p.is_dir()), None)
    if candidate is None:
        raise RuntimeError(f"run this notebook from src/kaggle_gpu (cwd is {HERE})")
    os.chdir(candidate)
sys.path.insert(0, str(Path.cwd()))

from kgpu import pdf_ocr, runner            # noqa: E402

print("cwd :", Path.cwd())
print("user:", pdf_ocr.kaggle_user())

In [ ]:
# ── THE JOB — computed from the parameters, validated exactly as a file job is ─
# ⚠️ `pdf_ocr.job` goes through `config._validate`, the same function `kaggle_config.json`
# jobs go through. Nothing is bypassed: the payload/parameters cross-check, the Kaggle title
# length rule and the quarters-must-be-YYYY-QQ rule all still fire, here, where they are free.
CFG = pdf_ocr.job(
    SYMBOL,
    exchange=EXCHANGE,
    periods=PERIODS,
    quarters=QUARTERS,
    allow_parent=ALLOW_PARENT,
    template=TEMPLATE,
    layers=LAYERS,
    compare=COMPARE,
    notes=NOTES,
    merge_statements=MERGE_INTO_CSV,
)

print("\n".join(pdf_ocr.describe(CFG)))

In [ ]:
# ── WHAT IT WOULD DO — touches nothing, spends nothing ────────────────────────
# The filings this selects are the filings the WORKER will open: `plan()` calls
# `FinancialsBuilder.documents()` here, so the payload cannot diverge from the worker's own
# choice. Read the list before paying for it.
runner.plan(CFG)

In [ ]:
# ── REHEARSE — the worker side, locally, no Kaggle quota ──────────────────────
# ⚠️ It does NOT run an OCR pass. What it proves is that the payload has every input the parse
# reads, under BOTH of Kaggle's mount layouts, and it prints the magnitude band `sane` will
# get. **An empty band is the warning to stop for**: `sane` fails open without one, and that
# is the documented way a run writes a wrong figure (CLAUDE.md §6-2-octodecies).
if REHEARSE:
    runner.rehearse(CFG)
else:
    print("skipped")

In [ ]:
# ── RUN — export, upload, push, wait, pull ────────────────────────────────────
# ⚠️ ONE CALL, and it blocks until Kaggle finishes. Budget: a filing that parses at layer 1
# takes ~1 min, one that defeats the whole cascade took 26-32 min when measured, and there is
# a QUEUE of ~5 min before either starts. `refresh_data=True` re-exports and re-uploads the
# payload every time — correct, because the filter above may have changed since the last run.
if RUN:
    if MERGE_INTO_CSV:
        print("MERGE_INTO_CSV is on: accepted statements are upserted into\n"
              "    raw_data/.../statements/ after the pull, with a backup "
              "and every changed cell printed.")
    EXIT = runner.run(CFG, refresh_data=True)
    print(f"\nexit {EXIT}   (0 = COMPLETE and pulled)")
else:
    EXIT = None
    print("RUN = False — nothing was pushed")

## The result

`kgpu` merges the run folder into `reports/pdf_ocr/<run_id>/`. The cell below reads the newest
one for this ticker.

**`compare()` scores every parsed cell against the statement CSV on disk**, so a verdict means:

| verdict | what it says |
|---|---|
| `REPRODUCED` | every cell, the winning LAYER, the unit and `publish_date` all match disk |
| `DIFFERS` | at least one of those moved — the run names which, with both figures |
| `ABSENT` | the cascade refused the statement; the log says why, layer by layer |
| *(refused)* | a cumulative income statement is not scored against a de-cumulated row — that would report every cell as changed |

⚠️ **A new `pdf` where disk holds `missing` is a RECOVERY, not a reproduction** — it has no
baseline to be scored against, so read the figures against the filing by hand before anything
is merged.

In [ ]:
# ── READ THE RUN FOLDER ───────────────────────────────────────────────────────
# ⚠️ `metadata.json` already carries the whole scorecard in `results` — one row per
# (period, report). Walking `documents/*.json` would re-derive it, and a second derivation is
# a second thing to keep in step.
import json

REPO = Path(runner.__file__).resolve().parents[3]
REPORTS = REPO / "reports" / "pdf_ocr"
pattern = f"*__{EXCHANGE.lower()}_{SYMBOL.lower()}__pdf_ocr"
folders = sorted(REPORTS.glob(pattern), key=lambda p: p.name)

if not folders:
    print(f"no run folder matching {pattern} under {REPORTS}")
else:
    LATEST = folders[-1]
    meta = json.loads((LATEST / "metadata.json").read_text(encoding="utf-8"))
    ocr = meta.get("environment", {}).get("ocr", {})
    inputs = meta.get("inputs", {})

    print(LATEST.name)
    print(f"  commit       : {meta.get('git_commit')}")
    print(f"  filter       : periods={inputs.get('periods')}  quarters={inputs.get('quarters')}")
    print(f"  template     : {inputs.get('template_requested') or 'resolved on the worker'}")
    # ⚠️ THE TWO OCR HALVES FAIL INDEPENDENTLY — detection is onnxruntime, recognition is
    # torch — so "the GPU was used" is two questions. `ORT-1` is a green run that was half on
    # the CPU because onnxruntime ADVERTISED a provider the session then could not create.
    print(f"  detection    : {(ocr.get('det_providers') or ['?'])[0]}"
          f"   (onnxruntime {ocr.get('onnxruntime')})")
    print(f"  recognition  : {ocr.get('recognizer_device')}")
    print(f"  stack        : {ocr.get('stack_fingerprint')}"
          f"{'  ⚠️ PIN VIOLATIONS: ' + str(ocr['pin_violations']) if ocr.get('pin_violations') else ''}")
    print()
    print(f"  {'period':10} {'report':18} {'layer':30} {'items':>5}  {'status':8} verdict")
    for r in meta.get("results", []):
        print(f"  {r['period']:10} {r['report']:18} {(r['layer'] or '—'):30} "
              f"{r['items']:>5}  {r['status']:8} {r['verdict']}")
    # ⚠️ `seconds` is the DOCUMENT's cost, repeated on each of its three report rows, so it is
    # summed per PERIOD. A set would also collapse two documents that took the same time.
    per_doc = {r["period"]: r["seconds"] for r in meta.get("results", [])}
    print(f"\n  parse: {sum(per_doc.values()) / 60:.1f} min over {len(per_doc)} document(s)")

In [ ]:
# ── THE LOG, when something needs explaining ──────────────────────────────────
# ⚠️ **READ THE FIRST REFUSAL, NOT THE LAST.** `_parse_cascaded` prints the DISTINCT reasons
# with the first layer that gave each, and a cascade's FINAL refusal names the hardest path
# tried rather than the blocking defect — the label `fx not mapped` sent this repo down a
# wrong diagnosis for two days that way (CLAUDE.md §6-2-duovicies).
if folders:
    log = (LATEST / "run.log").read_text(encoding="utf-8", errors="replace")
    hits = [ln for ln in log.splitlines()
            if "absent after" in ln or "reconcile:" in ln or "sane:" in ln]
    print("\n".join(hits) if hits else "no refusals — every statement was accepted")